# 02 – Cleaning

Clean and validate the raw data, save tidy files to `data/clean/`.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
raw_folder = Path("../data/raw")
clean_folder = Path("../data/clean")

clean_folder.mkdir(parents=True, exist_ok=True)

In [3]:
missions = pd.read_csv(raw_folder / "BFw_mission_data_daily.csv",
                       parse_dates=["mission_created_date"])

print(missions.shape)
missions.head()

(3223, 30)


,mission_created_date,mission_count_all,mission_count_ems,mission_count_ems_critical,mission_count_ems_critical_cpr,mission_count_fire,mission_count_technical_rescue,mission_count_rd1,mission_count_rd2,mission_count_rd3,...,response_time_fire_time_to_first_pump_std,response_time_fire_time_to_first_ladder_mean,response_time_fire_time_to_first_ladder_median,response_time_fire_time_to_first_ladder_std,response_time_fire_time_to_full_crew_mean,response_time_fire_time_to_full_crew_median,response_time_fire_time_to_full_crew_std,response_time_technical_rescue_mean,response_time_technical_rescue_median,response_time_technical_rescue_std
0,2018-01-01,2229,1781,1019,22,393,55,129,466,459,...,291.704676,905.597015,803.0,339.034729,842.269000,791.417,164.182813,799.720930,788.0,308.670060
1,2018-01-02,1240,1164,987,17,34,42,74,342,232,...,107.758334,585.571429,594.0,131.212260,690.270933,654.644,112.561205,872.527778,783.5,315.792562
2,2018-01-03,1354,1279,1081,21,32,43,71,372,219,...,110.773139,657.888889,624.0,149.065895,730.786450,739.173,95.878512,940.968750,971.0,283.249176
3,2018-01-04,1308,1243,1053,18,30,35,59,317,242,...,111.380240,662.347826,658.0,144.920859,747.760471,755.850,134.898659,803.222222,871.0,320.099664
4,2018-01-05,1269,1197,1033,22,38,33,79,355,237,...,92.698828,640.636364,584.5,200.938642,707.352125,709.355,111.396765,799.222222,804.0,252.471070


In [4]:
missions.info()

<class 'pandas.DataFrame'>
RangeIndex: 3223 entries, 0 to 3222
Data columns (total 30 columns):
 #   Column                                          Non-Null Count  Dtype         
---  ------                                          --------------  -----         
 0   mission_created_date                            3223 non-null   datetime64[us]
 1   mission_count_all                               3223 non-null   int64         
 2   mission_count_ems                               3223 non-null   int64         
 3   mission_count_ems_critical                      3223 non-null   int64         
 4   mission_count_ems_critical_cpr                  3223 non-null   int64         
 5   mission_count_fire                              3223 non-null   int64         
 6   mission_count_technical_rescue                  3223 non-null   int64         
 7   mission_count_rd1                               3223 non-null   int64         
 8   mission_count_rd2                               3223 non-nu

In [5]:
print("First day:", missions["mission_created_date"].min())
print("Last day: ", missions["mission_created_date"].max())
print("Duplicate dates:", missions["mission_created_date"].duplicated().sum())

all_days = pd.date_range(missions["mission_created_date"].min(),
                         missions["mission_created_date"].max())
missing_days = all_days.difference(missions["mission_created_date"])
print("Missing days:", len(missing_days))
print(missing_days)

First day: 2018-01-01 00:00:00
Last day:  2026-11-04 00:00:00
Duplicate dates: 0
Missing days: 7
DatetimeIndex(['2026-01-23', '2026-04-20', '2026-06-19', '2026-06-21',
               '2026-07-13', '2026-07-23', '2026-08-24'],
              dtype='datetime64[us]', freq=None)


In [6]:
columns = ["mission_count_ems", "response_time_ems_critical_median"]

summer = missions[missions["mission_created_date"].dt.month.isin([6, 7, 8])]

print(summer[columns].isna().sum())
print(summer.groupby(summer["mission_created_date"].dt.year).size())

mission_count_ems                    0
response_time_ems_critical_median    0
dtype: int64
mission_created_date
2018    92
2019    92
2020    92
2021    92
2022    92
2023    92
2024    92
2025    92
2026    87
dtype: int64


In [7]:
summer_ems = summer.set_index("mission_created_date")["mission_count_ems"]

print("Lowest summer days:")
print(summer_ems.nsmallest(5))
print("\nHighest summer days:")
print(summer_ems.nlargest(5))

Lowest summer days:
mission_created_date
2018-06-07     655
2019-07-07     976
2018-06-24     979
2018-07-10     985
2019-07-14    1011
Name: mission_count_ems, dtype: int64

Highest summer days:
mission_created_date
2026-06-28    1840
2026-06-29    1807
2026-06-27    1776
2026-06-30    1722
2026-07-02    1659
Name: mission_count_ems, dtype: int64


In [8]:
check_columns = ["mission_count_all", "mission_count_ems",
                 "mission_count_fire", "mission_count_technical_rescue"]

missions.set_index("mission_created_date").loc["2018-06-04":"2018-06-10", check_columns]

,mission_count_all,mission_count_ems,mission_count_fire,mission_count_technical_rescue
mission_created_date,,,,
2018-06-04,1367,1275,44,48
2018-06-05,1265,1171,40,52
2018-06-06,1323,1211,52,59
2018-06-07,724,655,31,37
2018-06-08,1374,1277,42,51
2018-06-09,1365,1266,60,38
2018-06-10,1269,1169,45,54


## Cleaning decisions – Fire Brigade data
- Kept only June–August, 2018–2026.
- Kept the date, the daily number of emergency medical (EMS) missions and CPR-alert missions (for an optional extra analysis).
- **Excluded 7 June 2018 as a suspected reporting anomaly:** EMS missions fell by about 47% compared with the average of the two neighbouring days (total missions −46%, fire and technical-rescue missions about −33%). Incomplete recording is a plausible explanation but cannot be confirmed from the data. The original value remains in the raw file.
- **5 summer days are missing in 2026** (19 Jun, 21 Jun, 13 Jul, 23 Jul, 24 Aug). They are absent from the source data and are not filled in.

In [9]:
missions_clean = summer[["mission_created_date", "mission_count_ems",
                         "mission_count_ems_critical_cpr"]].copy()

missions_clean = missions_clean.rename(columns={
    "mission_created_date": "date",
    "mission_count_ems": "ems_missions",
    "mission_count_ems_critical_cpr": "cpr_alert_missions"
})

missions_clean = missions_clean[missions_clean["date"] != pd.Timestamp("2018-06-07")]
missions_clean = missions_clean.reset_index(drop=True)

print(missions_clean.shape)
missions_clean.head()

(822, 3)


,date,ems_missions,cpr_alert_missions
0,2018-06-01,1357,20
1,2018-06-02,1188,11
2,2018-06-03,1194,14
3,2018-06-04,1275,15
4,2018-06-05,1171,18


In [10]:
missions_clean.to_csv(clean_folder / "missions_summer_clean.csv", index=False)
print("Saved.")

Saved.


## Weather: DWD station Berlin-Tempelhof

In [11]:
import zipfile

def read_dwd_zip(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        data_file = [name for name in z.namelist() if name.startswith("produkt_klima_tag")][0]
        with z.open(data_file) as f:
            df = pd.read_csv(f, sep=";", skipinitialspace=True)
    df.columns = df.columns.str.strip()
    return df

In [12]:
weather_hist = read_dwd_zip(raw_folder / "tageswerte_KL_00433_19480101_20251231_hist.zip")
weather_recent = read_dwd_zip(raw_folder / "tageswerte_KL_00433_akt.zip")

print("Historical:", weather_hist.shape)
print("Recent:    ", weather_recent.shape)

Historical: (28490, 19)
Recent:     (550, 19)


In [13]:
# Check the overlap between the historical and recent files BEFORE removing duplicates
overlap = weather_hist[["MESS_DATUM", "TXK"]].merge(
    weather_recent[["MESS_DATUM", "TXK"]],
    on="MESS_DATUM",
    suffixes=("_hist", "_recent"),
    validate="one_to_one"
)

different = overlap[overlap["TXK_hist"] != overlap["TXK_recent"]]

print("Overlapping dates:", len(overlap))
print("Different temperatures:", len(different))

if not different.empty:
    display(different.head(10))
    raise ValueError("Historical and recent temperatures differ. Investigate before deduplication.")

# Historical values take priority
weather = pd.concat([weather_hist, weather_recent], ignore_index=True)
weather = weather.drop_duplicates(subset="MESS_DATUM", keep="first")
weather["date"] = pd.to_datetime(weather["MESS_DATUM"].astype(str), format="%Y%m%d")

assert weather["date"].is_unique

print("First day:", weather["date"].min())
print("Last day:", weather["date"].max())

Overlapping dates: 242
Different temperatures: 0
First day: 1948-01-01 00:00:00
Last day: 2026-11-04 00:00:00


In [14]:
weather_clean = weather[["date", "TXK", "QN_4"]].copy()
weather_clean = weather_clean.rename(columns={"TXK": "tmax", "QN_4": "quality"})

weather_clean["tmax"] = weather_clean["tmax"].replace(-999, float("nan"))

month = weather_clean["date"].dt.month
day = weather_clean["date"].dt.day
in_scope = (weather_clean["date"].dt.year >= 2018) & (month.isin([6, 7, 8]) | ((month == 5) & (day >= 25)))

weather_clean = weather_clean[in_scope].reset_index(drop=True)

print(weather_clean.shape)
weather_clean.head()

(891, 3)


,date,tmax,quality
0,2018-05-25,26.9,9
1,2018-05-26,27.3,9
2,2018-05-27,28.2,9
3,2018-05-28,32.1,9
4,2018-05-29,32.2,9


In [15]:
print("Missing temperatures:", weather_clean["tmax"].isna().sum())

print("\nDays per year:")
print(weather_clean.groupby(weather_clean["date"].dt.year).size())

summer_weather = weather_clean[weather_clean["date"].dt.month.isin([6, 7, 8])]
print("\nHot days (>= 30°C) per summer:")
print((summer_weather["tmax"] >= 30).groupby(summer_weather["date"].dt.year).sum())

print("\nQuality level per year:")
print(weather_clean.groupby([weather_clean["date"].dt.year, "quality"]).size().unstack(fill_value=0))

Missing temperatures: 0

Days per year:
date
2018    99
2019    99
2020    99
2021    99
2022    99
2023    99
2024    99
2025    99
2026    99
dtype: int64

Hot days (>= 30°C) per summer:
date
2018    25
2019    25
2020    16
2021    13
2022    21
2023    11
2024    14
2025    11
2026    21
Name: tmax, dtype: int64

Quality level per year:
quality  1   3   9
date              
2018     0   0  99
2019     0   0  99
2020     0   0  99
2021     0   0  99
2022     0   0  99
2023     0   0  99
2024     0   0  99
2025     0   0  99
2026     8  91   0


## Cleaning decisions – weather data
- Combined the historical file (to 2025) and the recent file (2026). The 242 overlapping days were checked and had identical temperatures; the historical value was kept.
- Kept daily maximum temperature (`tmax`) and DWD's quality level (`quality`).
- Kept June–August plus 25–31 May (for the lag analysis), 2018–2026.
- No missing temperatures in scope.
- Quality levels: 2018–2025 come from DWD's historical archive with quality level 9 ("not all parameters corrected"). 2026 comes from the recent archive with levels 3 (automatic control and correction) and 1 (formal checks only), so 2026 is preliminary. Level 9 does not mean fully finalised; that would be level 10.

In [16]:
weather_clean.to_csv(clean_folder / "weather_clean.csv", index=False)
print("Saved.")

Saved.


## Heat warnings: DWD historical heat alerts

In [17]:
warning_tables = []

for year in range(2018, 2027):
    df = pd.read_csv(raw_folder / f"heat_alerts_{year}.csv", encoding="utf-8-sig")
    warning_tables.append(df)

warnings_all = pd.concat(warning_tables)

print(warnings_all.shape)
warnings_all.head()

(32884, 5)


,Landkreis.Kennung,Datum,Warnstufe,Name,Bundesland
0,ABI,2018-05-27,1,Kreis Anhalt-Bitterfeld,Sachsen-Anhalt
1,ABX,2018-05-27,1,Kreis und Stadt Aschaffenburg,Bayern
2,AKX,2018-05-27,1,Kreis Altenkirchen (Westerwald),Rheinland-Pfalz
3,BIX,2018-05-27,1,Stadt Bielefeld,Nordrhein-Westfalen
4,BLK,2018-05-27,1,Burgenlandkreis,Sachsen-Anhalt


In [18]:
warnings_all[warnings_all["Name"].str.contains("Berlin")][["Landkreis.Kennung", "Name", "Bundesland"]].drop_duplicates()

,Landkreis.Kennung,Name,Bundesland
12,BXX,Berlin,Berlin


In [19]:
warnings_clean = warnings_all[warnings_all["Landkreis.Kennung"] == "BXX"].copy()
warnings_clean = warnings_clean[["Datum", "Warnstufe"]].rename(columns={
    "Datum": "date",
    "Warnstufe": "warning_level"
})
warnings_clean["date"] = pd.to_datetime(warnings_clean["date"])

print("All Berlin warning days:", len(warnings_clean))

month = warnings_clean["date"].dt.month
day = warnings_clean["date"].dt.day
in_scope = month.isin([6, 7, 8]) | ((month == 5) & (day >= 25))

print("Outside scope (dropped):")
print(warnings_clean[~in_scope])

warnings_clean = warnings_clean[in_scope].reset_index(drop=True)

All Berlin warning days: 99
Outside scope (dropped):
           date  warning_level
3210 2024-09-03              1
3273 2024-09-04              1


In [20]:
print("Duplicate dates:", warnings_clean["date"].duplicated().sum())
print(warnings_clean.shape)

print("\nWarning levels:")
print(warnings_clean["warning_level"].value_counts())

summer_warnings = warnings_clean[warnings_clean["date"].dt.month.isin([6, 7, 8])]
print("\nWarning days per summer:")
print(summer_warnings.groupby(summer_warnings["date"].dt.year).size())

Duplicate dates: 0
(97, 2)

Warning levels:
warning_level
1    88
3     9
Name: count, dtype: int64

Warning days per summer:
date
2018    17
2019    16
2020     8
2021     5
2022    17
2023     7
2024     5
2025     5
2026    12
dtype: int64


## Cleaning decisions – heat warnings
- Combined the 9 yearly files (2018–2026) and kept only Berlin (district ID `BXX`; Berlin is a single warning district).
- Kept the date and warning level: 1 = strong heat stress (perceived temperature above about 32°C for at least two days, with warm nights), 3 = extreme heat stress (perceived temperature above 38°C).
- Kept June–August plus 25–31 May; dropped 2 warnings in September 2024.
- The table only contains warning days. Days without a warning are not listed and will be treated as "no warning" when joining.
- DWD notes that warning criteria have changed over time, so the data is not suited for trend analysis.

In [21]:
warnings_clean.to_csv(clean_folder / "heat_warnings_clean.csv", index=False)
print("Saved.")

Saved.


## Combine all three datasets

In [22]:
missions = pd.read_csv(clean_folder / "missions_summer_clean.csv", parse_dates=["date"])
weather = pd.read_csv(clean_folder / "weather_clean.csv", parse_dates=["date"])
warnings = pd.read_csv(clean_folder / "heat_warnings_clean.csv", parse_dates=["date"])

print(missions.shape, weather.shape, warnings.shape)

(822, 3) (891, 3) (97, 2)


In [23]:
daily = weather.merge(missions, on="date", how="left")
daily = daily.merge(warnings, on="date", how="left")

print(daily.shape)
print(daily.isna().sum())

(891, 6)
date                    0
tmax                    0
quality                 0
ems_missions           69
cpr_alert_missions     69
warning_level         794
dtype: int64


In [24]:
daily["warning_level"] = daily["warning_level"].fillna(0).astype(int)
daily["heat_warning"] = (daily["warning_level"] > 0).astype(int)
daily["hot_day"] = (daily["tmax"] >= 30).astype(int)

daily["year"] = daily["date"].dt.year
daily["month"] = daily["date"].dt.month
daily["weekday"] = daily["date"].dt.day_name()
daily["weekday_num"] = daily["date"].dt.dayofweek
daily["in_summer"] = daily["month"].isin([6, 7, 8]).astype(int)

daily.head()

,date,tmax,quality,ems_missions,cpr_alert_missions,warning_level,heat_warning,hot_day,year,month,weekday,weekday_num,in_summer
0,2018-05-25,26.9,9,NaN,NaN,0,0,0,2018,5,Friday,4,0
1,2018-05-26,27.3,9,NaN,NaN,0,0,0,2018,5,Saturday,5,0
2,2018-05-27,28.2,9,NaN,NaN,1,1,0,2018,5,Sunday,6,0
3,2018-05-28,32.1,9,NaN,NaN,1,1,1,2018,5,Monday,0,0
4,2018-05-29,32.2,9,NaN,NaN,1,1,1,2018,5,Tuesday,1,0


In [25]:
summer_days = daily[daily["in_summer"] == 1]

print("Summer days:", len(summer_days))
print("Summer days without mission data:", summer_days["ems_missions"].isna().sum())
print(summer_days[summer_days["ems_missions"].isna()][["date", "tmax", "hot_day"]])

print("\nHot days:", summer_days["hot_day"].sum(), "| Warning days:", summer_days["heat_warning"].sum())
print(pd.crosstab(summer_days["hot_day"], summer_days["heat_warning"], margins=True))

Summer days: 828
Summer days without mission data: 6
          date  tmax  hot_day
13  2018-06-07  28.0        0
817 2026-06-19  35.3        1
819 2026-06-21  30.8        1
841 2026-07-13  29.8        0
851 2026-07-23  22.3        0
883 2026-08-24  21.6        0

Hot days: 157 | Warning days: 92
heat_warning    0   1  All
hot_day                   
0             664   7  671
1              72  85  157
All           736  92  828


## Combined table: `daily_summer.csv`
- One row per day: 25 May – 31 August, 2018–2026 (891 rows).
- Base = weather data (complete), with missions and warnings joined by date.
- `in_summer = 0` marks the late-May helper days (only used for lag analysis).
- 6 summer days have no mission data (5 missing in 2026 + 7 June 2018 excluded) and are left empty.
- `hot_day` = 1 if maximum temperature ≥ 30°C; `heat_warning` = 1 if DWD issued a heat warning for Berlin.

In [26]:
# FINAL DATA INTEGRITY CHECKS
# The fixed numbers describe the data snapshot downloaded on 5 November 2026.

# Unique dates
assert missions_clean["date"].is_unique
assert weather_clean["date"].is_unique
assert warnings_clean["date"].is_unique
assert daily["date"].is_unique

# Expected sizes
assert len(missions_clean) == 822
assert len(weather_clean) == 891
assert len(warnings_clean) == 97
assert len(daily) == 891

# Essential values present
assert missions_clean[["date", "ems_missions", "cpr_alert_missions"]].notna().all().all()
assert weather_clean[["date", "tmax"]].notna().all().all()
assert daily["tmax"].notna().all()
assert daily["warning_level"].notna().all()

# Plausible values
assert (missions_clean["ems_missions"] >= 0).all()
assert missions_clean["cpr_alert_missions"].between(0, missions_clean["ems_missions"]).all()

# Every mission date has weather data
assert missions_clean["date"].isin(weather_clean["date"]).all()

# Scope
assert daily["date"].dt.month.isin([5, 6, 7, 8]).all()
summer_check = daily[daily["in_summer"] == 1]
assert len(summer_check) == 828
assert summer_check["ems_missions"].isna().sum() == 6

print("All data integrity checks passed.")

All data integrity checks passed.


In [27]:
daily.to_csv(clean_folder / "daily_summer.csv", index=False)
print("Saved.")

Saved.
